# 02. FAISS `load_local` 과 pickle 역직렬화 취약점 (OWASP LLM04 / ASI04)

| 항목 | 내용 |
|---|---|
| OWASP | **LLM04:2026 Supply Chain** (외부에서 온 모델·데이터·파일을 신뢰) / **ASI04 Agentic Supply Chain** |
| 원본 위치 | `allow_dangerous_deserialization=True` 를 쓰는 노트북 10곳 (`01RAG/*`, `04LangChainTool_Agent/20-2*`, `05LangGraph/01·04·05·06` 등) |
| 위험도 | 높음 (임의 코드 실행) — 단, **변조된 인덱스 파일을 불러올 때만** |
| LLM 호출 | 없음 — 가짜 임베딩(`DeterministicFakeEmbedding`)을 사용하므로 API 키·비용이 필요 없습니다 |

## 핵심 개념
- `FAISS.save_local()` 은 `index.faiss`(벡터)와 **`index.pkl`(문서 + ID 매핑, pickle 형식)** 두 파일을 만듭니다.
- **pickle 은 불러오는 순간 코드를 실행할 수 있는 형식**입니다. 그래서 LangChain 이 `allow_dangerous_deserialization=True` 를 직접 입력해야 불러오도록 막아 둡니다.
- 직접 만든 파일만 로드하면 괜찮다는 말은 맞지만, **"내 파일이 그대로인지"를 확인하지 않으면** 공유 폴더, 다운로드, 팀원이 올린 `db/` 등에서 바뀐 파일을 그대로 실행하게 됩니다.

## 이 위험은 언제 현실이 되나요?

학습 코드처럼 **바로 앞 셀에서 내가 만든 인덱스를 다시 불러오는 경우에는 위험이 낮습니다.** 공격자가 내 노트북 폴더의 파일을 바꿀 수 있어야 하기 때문입니다. 위험이 현실이 되는 것은 아래 두 경우입니다.

| 시나리오 | 예 | 알맞은 대책 |
|---|---|---|
| **A. 남이 만든 인덱스를 받아 쓴다** (가장 흔함) | GitHub 예제 저장소의 `db/` 폴더, 공유 드라이브, 블로그·강의 첨부 파일 | **열지 말고 원본 문서로 다시 만든다** (3장) |
| **B. 내가 만든 인덱스를 다른 사람도 쓸 수 있는 곳에 두고 나중에 불러온다** | 팀 공유 서버, 빌드 서버에서 만들어 운영 서버로 복사 | **저장할 때 서명, 불러오기 전에 검증** (4장) |

> 시나리오 A에서는 서명이 소용없습니다. 남이 만든 파일에는 내 서명을 붙일 수 없기 때문입니다. **대책이 상황마다 다르다**는 점이 이 노트북의 핵심입니다.

## 이 노트북의 흐름
1. 정상 인덱스 만들기 → Before 방식으로 로드
2. `index.pkl` 을 변조해 공격 재현 (무해한 페이로드: 마커 파일 생성)
3. After ① (시나리오 A): 받은 인덱스는 열지 않고 **원본 문서로 다시 만들기**
4. After ② (시나리오 B): 저장 시 **HMAC(Hash-based Message Authentication Code) 서명**, 로드 전에 **검증**
5. 같은 공격 재시도 → 차단 확인

In [ ]:
import hashlib
import hmac
import os
import pickle
import secrets
import tempfile
from pathlib import Path

from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import DeterministicFakeEmbedding

# 가짜 임베딩: 실습용 (실제 의미 검색은 안 됨, 파일 구조만 같음)
embeddings = DeterministicFakeEmbedding(size=64)

# 실습용 임시 폴더 (노트북 종료 후 자동 정리되지 않으므로 마지막 셀에서 삭제)
WORK_DIR = Path(tempfile.mkdtemp(prefix="faiss_demo_"))
DB_DIR = WORK_DIR / "menu_db"
MARKER = WORK_DIR / "PWNED.txt"      # 공격이 성공하면 생기는 표식 파일
print("작업 폴더:", WORK_DIR)

## 1. Before — 원본 방식

정상 인덱스를 만들고 `allow_dangerous_deserialization=True` 로 그대로 로드합니다.

In [ ]:
texts = ["스테이크는 레드 와인과 어울립니다", "파스타는 화이트 와인과 어울립니다", "치즈는 로제 와인과 어울립니다"]
FAISS.from_texts(texts, embeddings).save_local(str(DB_DIR))
print("저장된 파일:", sorted(p.name for p in DB_DIR.iterdir()))


def load_before(db_path: str):
    """[취약] 파일이 바뀌었는지 확인하지 않고 pickle 을 그대로 로드"""
    return FAISS.load_local(db_path, embeddings, allow_dangerous_deserialization=True)


db = load_before(str(DB_DIR))
print("정상 로드:", db.index.ntotal, "개 벡터")

## 2. 공격 재현 — `index.pkl` 변조

공격자가 `index.pkl` 을 바꿔치기했다고 가정합니다. 위 표의 **시나리오 A**라면 "내려받은 `db/` 폴더가 처음부터 이런 파일"인 상황이고, **시나리오 B**라면 "공유 서버에 둔 내 인덱스를 누군가 바꾼" 상황입니다.
pickle 은 객체를 복원할 때 `__reduce__` 가 지정한 **함수를 호출**합니다. 여기서는 무해하게 **마커 파일을 만드는 함수**를 지정했습니다.
실제 공격이라면 이 자리에 `os.system(...)`, 키 탈취, 백도어 설치가 들어갑니다.

In [ ]:
class Evil:
    """역직렬화(unpickle)될 때 Path.write_text 를 호출하게 만드는 악성 객체 (무해한 예시)"""
    def __reduce__(self):
        return (Path.write_text, (MARKER, "이 파일이 보이면 pickle 로드 중 코드가 실행된 것입니다"))


def tamper(db_path: Path):
    """index.pkl 을 악성 pickle 로 교체 (공격자 역할)"""
    (db_path / "index.pkl").write_bytes(pickle.dumps(Evil()))


tamper(DB_DIR)
print("변조 전 마커 존재:", MARKER.exists())

try:
    load_before(str(DB_DIR))
except Exception as e:       # 형식이 맞지 않아 오류가 나도, 코드는 이미 실행된 뒤
    print("로드 중 오류:", type(e).__name__)

print("변조 후 마커 존재:", MARKER.exists(), "  <-- True 면 공격 성공(임의 코드 실행)")
assert MARKER.exists(), "취약점이 재현되지 않았습니다"

> **결과 해석**: 오류가 나더라도 **이미 코드는 실행된 뒤**입니다. 오류가 안 나게 만드는 것도 공격자에게는 쉽습니다.
> "예외가 발생하지 않았다" 또는 "정상 동작한다"는 안전의 증거가 아닙니다.

## 3. After ① — (시나리오 A) 받은 인덱스는 열지 않고 원본 문서로 다시 만들기

가장 단순하고 확실한 방법입니다. **pickle 파일을 아예 열지 않으면** 그 안에 무엇이 들어 있든 실행될 수 없습니다.

- 예제 저장소에 `db/` 폴더와 원본 문서(`data/*.pdf` 등)가 함께 있다면, `db/` 는 무시하고 원본 문서로 인덱스를 새로 만듭니다.
- 새로 만든 인덱스는 **내가 만든 것**이므로 이후에는 내 폴더에서 불러와도 됩니다.
- 비용: 실제 임베딩 모델을 쓰면 다시 만드는 데 시간과 API 비용이 듭니다. 문서가 아주 많아 매번 다시 만들기 어렵다면 4장(서명) 방식을 씁니다.

In [ ]:
def build_from_source(source_texts: list[str], save_to: Path) -> FAISS:
    """[개선 ①] 원본 문서로 인덱스를 새로 만든다. 받은 index.pkl 은 열지 않는다."""
    db = FAISS.from_texts(source_texts, embeddings)
    db.save_local(str(save_to))          # 이제부터는 '내가 만든' 인덱스
    return db


MARKER.unlink(missing_ok=True)           # 앞 실험의 표식 제거
downloaded_db = DB_DIR                   # 2장에서 변조된 폴더 = "인터넷에서 받은 db/" 라고 가정

# 받은 db/ 는 무시하고, 함께 받은 원본 문서(texts)로 다시 만든다
my_db = build_from_source(texts, WORK_DIR / "my_db")
print("다시 만든 인덱스:", my_db.index.ntotal, "개 벡터")
print("마커 존재:", MARKER.exists(), "  <-- False: 받은 pickle 을 열지 않았으므로 코드가 실행되지 않음")
assert not MARKER.exists()

## 4. After ② — (시나리오 B) 저장 시 서명(HMAC), 로드 전 검증

인덱스가 크고 내가 직접 만들지만, **다른 사람도 쓸 수 있는 곳**에 두는 경우입니다.

수정 원칙
1. 인덱스를 **저장할 때** 파일 내용에 대한 HMAC 서명을 만들어 둡니다. (SHA-256 사용)
2. **로드하기 전에** 서명을 다시 계산해 비교합니다. 다르면 `load_local` 자체를 호출하지 않습니다.
3. 서명 키는 코드에 쓰지 않고 **환경 변수**에서 읽습니다. (공격자가 파일을 바꿔도 키가 없으면 서명을 만들 수 없음)
4. 비교는 `hmac.compare_digest` 로 합니다. (타이밍 공격 방지)

> 해시(SHA-256)만 저장하면 공격자가 파일과 해시를 함께 바꿀 수 있어서, **비밀 키가 있는 HMAC** 이 필요합니다.

In [ ]:
SIGN_FILES = ("index.faiss", "index.pkl")     # 서명 대상 파일
SIG_NAME = "index.sig"                        # 서명 저장 파일
KEY_ENV = "FAISS_INTEGRITY_KEY"               # 서명 키를 담을 환경 변수 이름

# 실습용 임시 키: 이 노트북 안에서만 쓰는 값이다. 실제 코드(secure_modular, 05 보안판)는 키가 없으면 임시 키를 만들지 않고 즉시 중단한다.
os.environ.setdefault(KEY_ENV, secrets.token_hex(32))


def _signature(db_path: Path) -> str:
    """두 파일의 내용을 이어서 HMAC-SHA256 서명을 계산"""
    key = os.environ[KEY_ENV].encode()
    mac = hmac.new(key, digestmod=hashlib.sha256)
    for name in SIGN_FILES:
        mac.update(name.encode())                 # 파일 이름도 포함 (파일 바꿔치기 방지)
        mac.update((db_path / name).read_bytes())
    return mac.hexdigest()


def save_signed(db: FAISS, db_path: str) -> None:
    """인덱스를 저장하고 서명을 함께 기록"""
    db.save_local(db_path)
    (Path(db_path) / SIG_NAME).write_text(_signature(Path(db_path)), encoding="utf-8")


def load_after(db_path: str) -> FAISS:
    """[개선] 서명이 일치할 때만 load_local 을 호출"""
    p = Path(db_path)
    sig_file = p / SIG_NAME
    if not sig_file.exists():
        raise PermissionError("서명 파일이 없어 로드를 거부합니다")
    if not hmac.compare_digest(sig_file.read_text(encoding="utf-8").strip(), _signature(p)):
        raise PermissionError("서명이 일치하지 않아 로드를 거부합니다 (파일이 변조되었을 수 있음)")
    # 검증을 통과한 파일만 pickle 역직렬화를 허용
    return FAISS.load_local(str(p), embeddings, allow_dangerous_deserialization=True)

## 5. 같은 공격을 다시 시도

In [ ]:
SAFE_DIR = WORK_DIR / "menu_db_signed"
MARKER.unlink(missing_ok=True)                    # 앞 실험의 표식 제거
save_signed(FAISS.from_texts(texts, embeddings), str(SAFE_DIR))

# (1) 정상 파일은 문제없이 로드된다
ok = load_after(str(SAFE_DIR))
print("정상 로드:", ok.index.ntotal, "개 벡터")

# (2) 같은 방식으로 index.pkl 변조 → 로드 시도
tamper(SAFE_DIR)
try:
    load_after(str(SAFE_DIR))
    blocked = False
except PermissionError as e:
    blocked = True
    print("차단:", e)

print("마커 존재:", MARKER.exists(), "  <-- False 여야 안전")
assert blocked and not MARKER.exists(), "변조 파일이 로드되었습니다"
print("검증 통과: 변조된 인덱스는 pickle 이 실행되기 전에 거부됨")

In [ ]:
# (3) 공격자가 서명 파일까지 바꿔도 키를 모르면 맞는 서명을 만들 수 없다
(SAFE_DIR / SIG_NAME).write_text(hashlib.sha256((SAFE_DIR / "index.pkl").read_bytes()).hexdigest())
try:
    load_after(str(SAFE_DIR))
    blocked2 = False
except PermissionError as e:
    blocked2 = True
    print("차단:", e)
assert blocked2 and not MARKER.exists()
print("검증 통과: 서명 위조도 차단")

In [ ]:
# 실습 폴더 정리
import shutil

shutil.rmtree(WORK_DIR, ignore_errors=True)
print("정리 완료:", not WORK_DIR.exists())

## 6. 바뀐 점 요약

| 구분 | Before | After ① 다시 만들기 (시나리오 A) | After ② 서명 검증 (시나리오 B) |
|---|---|---|---|
| 받은/저장된 `index.pkl` | 그대로 역직렬화 | **열지 않음** | 서명이 맞을 때만 역직렬화 |
| 변조 파일 | 로드 시 코드 실행 | 영향 없음 | `load_local` 호출 전에 거부 |
| 필요한 것 | — | 원본 문서 | 비밀 키 (`FAISS_INTEGRITY_KEY`, 환경 변수) |
| 비용 | — | 다시 임베딩하는 시간·비용 | 저장 시 서명 1회 |

## 7. 남는 위험과 더 안전한 대안
- **키가 유출되거나 공격자가 빌드/저장 단계를 장악**하면 서명도 믿을 수 없습니다. 키는 `.env`/비밀 관리 도구에 두고 저장소에 올리지 마세요.
- **가장 안전한 방법은 pickle 자체를 쓰지 않는 것**입니다. 원본 문서(PDF, DOCX 등)에서 **시작할 때 인덱스를 다시 만들거나**, pickle 을 쓰지 않는 벡터 저장소(Chroma 등) 사용을 검토하세요.
- 외부에서 받은 인덱스(다운로드, 공유 폴더)는 **절대** `allow_dangerous_deserialization=True` 로 열지 마세요.
- `db/` 폴더를 Git 에 올리는 경우, 누가 수정할 수 있는지(PR 리뷰 여부)도 공급망 위험입니다.